# Day 4 lab: File formats and data layout

**Goal:** Pick a format, codec and partition layout for a workload, and prove each choice with plans, file counts and read-task counts.

**Time:** about 2 h 30 min. Checklist: 2.1–2.10.

Put [`lab_utils.py`](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/lab_utils.py) next to this notebook, run the setup cell, then work through the tasks in order. Write your code in the empty cell under each task and your observations in the notes cell. Keep the Spark UI open (the setup cell prints its address).

In [ ]:
from lab_utils import *

spark = get_spark()

## 📘 Code along

Type the code yourself instead of pasting it, run every cell, and keep the Spark UI open. Swap in `spark = get_spark()` from the starter kit wherever an example builds its own session.

### Task 1. Chapter 7: Load and Behold — 20 min

Type the reading, writing and options sections of the official guide. The chapter reads `../data/employees.*` files that are not shipped with it, so use the official sample files from `examples/src/main/resources` instead (download each with the Raw button): `people.csv` (semicolon-delimited, so `option("delimiter", ";")`), `people.json`, `users.parquet` and `users.orc`. Write each back out as CSV, Parquet and ORC, and try the gzip compression option.

[Chapter 7: Load and Behold](https://spark.apache.org/docs/latest/api/python/user_guide/loadandbehold.html) · [Sample files on GitHub](https://github.com/apache/spark/tree/v4.2.0/examples/src/main/resources) · [Notebook source on GitHub](https://github.com/apache/spark/blob/v4.2.0/python/docs/source/user_guide/loadandbehold.ipynb)

In [ ]:
# Task 1: your code here


*Task 1 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 2. The official datasource.py example — 25 min

Type `basic_datasource_example` and `parquet_schema_merging_example` into your notebook with the downloaded sample files (skip JDBC, Hive and XML): generic `load` / `save`, `partitionBy`, `bucketBy` with `saveAsTable`, and two batches under `key=1` and `key=2` read back with `mergeSchema`. After every write, list the output folder: one folder per partition value, bucket files, part files and `_SUCCESS`.

[The official datasource.py example](https://github.com/apache/spark/blob/v4.2.0/examples/src/main/python/sql/datasource.py) · [Data sources guide](https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html)

In [ ]:
# Task 2: your code here


*Task 2 notes: what you predicted, what the plan and the Spark UI showed.*

## ✍️ Exercises

Write these from scratch in the cells below, using the patterns you just typed. Each one ends with a Check you can verify yourself; open the reference solution only after you have a working answer.

### Task 3. Exercise 1: one dataset in five formats — 30 min

Take 1,000,000 rows of `data/orders` and write them as CSV (with header), JSON, Parquet, ORC and Avro. Record folder size and write time, then time `agg(F.sum("amount"))` on each with `run()`. Expect JSON largest, CSV slowest to write, Parquet and ORC smallest and fastest to read; ORC may be smaller than Parquet because Spark 4 uses zstd for ORC and snappy for Parquet. Check (1,000,000 rows): JSON about 200 MB, CSV about 100 MB, Avro about 60 MB, Parquet about 50 MB and ORC about 30 MB, and Parquet reads about 5× faster than CSV. Proves 2.1, 2.2, 2.10.

In [ ]:
# Task 3: your code here


*Task 3 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 4. Exercise 2: datasource.py's layouts applied to orders — 30 min

(1) Write orders with `partitionBy("order_date")`, then filter one date and `amount > 99` and read the scan node. (2) Like `parquet_schema_merging_example`, write 1,000 orders to `layout/batches/batch=1` and 1,000 with an extra `discount` column to `layout/batches/batch=2`, then read the parent folder with `mergeSchema`. 

**Check:** 30 date folders; `PartitionFilters` holds your date and `PushedFilters` holds `GreaterThan(amount,99.0)`; the merged DataFrame has `batch` and `discount` columns, and `discount` is null for every batch-1 row. Proves 2.8, 2.4, 2.3.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day04.py)

In [ ]:
# Task 4: your code here


*Task 4 notes: what you predicted, what the plan and the Spark UI showed.*

## 🔬 Prove it

Short, focused checks of today's checklist questions in the plan and the Spark UI.

### Task 5. Create the small file problem, then fix it — 25 min

Write orders with `repartition(200)` and `partitionBy("order_date")`: up to 6,000 tiny files. Time a full read and count its read tasks. Fix it with `repartition("order_date")` before `partitionBy` (one file per folder), or `coalesce(n)` with the `maxRecordsPerFile` option; compare file counts, read tasks and time. Proves 2.5.

In [ ]:
# Task 5: your code here


*Task 5 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 6. Splittable or not — 20 min

Write a CSV of about 200 MB as one file (`coalesce(1)`), once plain and once with `option("compression", "gzip")`. Read both and compare `rdd.getNumPartitions()`: plain splits across tasks, gzip is 1. Predict the plain count with the formula min(128 MB, max(4 MB, bytes ÷ cores)) and check; set `spark.sql.files.maxPartitionBytes` to `32m` and predict again. Proves 2.9.

In [ ]:
# Task 6: your code here


*Task 6 notes: what you predicted, what the plan and the Spark UI showed.*